# Module 4: Feature Engineering, Transformation & Selection

This notebook runs the leak-free Module 4 feature pipeline and writes the resulting CSV files directly into the relative path `data/processed` (with automatic path resolution if the notebook is run from `notebooks/` or the project root):
- `data/processed/X_train_selected.csv`
- `data/processed/y_train_log.csv`
- `data/processed/X_test_selected.csv`

In [1]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

# Resolve relative paths cleanly whether run from project root or notebooks/
if os.path.exists("src"):
    SRC_DIR = "src"
    DATA_PROCESSED_DIR = os.path.join("data", "processed")
    DATA_RAW_DIR = os.path.join("data", "raw")
elif os.path.exists("../src"):
    SRC_DIR = "../src"
    DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
    DATA_RAW_DIR = os.path.join("..", "data", "raw")
else:
    SRC_DIR = "src"
    DATA_PROCESSED_DIR = os.path.join("data", "processed")
    DATA_RAW_DIR = os.path.join("data", "raw")

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import numpy as np
import pandas as pd
import feature_engineering as fe

print(f"Using SRC_DIR: {SRC_DIR}")
print(f"Target output directory: {DATA_PROCESSED_DIR}")

Using SRC_DIR: ../src
Target output directory: ..\data\processed


In [2]:
# Load cleaned datasets from data/processed (fallback to data/raw if needed)
train_cleaned_path = os.path.join(DATA_PROCESSED_DIR, "train_cleaned.csv")
test_cleaned_path = os.path.join(DATA_PROCESSED_DIR, "test_cleaned.csv")

if os.path.exists(train_cleaned_path):
    train_df = pd.read_csv(train_cleaned_path)
    test_df = pd.read_csv(test_cleaned_path) if os.path.exists(test_cleaned_path) else None
    print(f"Loaded cleaned train data: {train_df.shape}")
else:
    raw_train_path = os.path.join(DATA_RAW_DIR, "train.csv")
    raw_test_path = os.path.join(DATA_RAW_DIR, "test.csv")
    if not os.path.exists(raw_train_path):
        raise FileNotFoundError(
            f"Neither {train_cleaned_path} nor {raw_train_path} was found."
        )
    train_df = pd.read_csv(raw_train_path)
    test_df = pd.read_csv(raw_test_path) if os.path.exists(raw_test_path) else None
    print(f"Loaded raw train data (fallback): {train_df.shape}")

if test_df is not None:
    print(f"Loaded test data: {test_df.shape}")

Loaded cleaned train data: (1458, 94)
Loaded test data: (1459, 93)


In [3]:
# Step 1: Feature construction (leak-free)
train_eng, test_eng = fe.construct_derived_features(
    train_df=train_df,
    test_df=test_df,
    target_col="SalePrice"
)

# Step 2: Encoding, skew transformation & scaling
X_train_enc, y_train_log, X_test_enc = fe.encode_and_transform_features(
    train_df=train_eng,
    test_df=test_eng,
    target_col="SalePrice",
    skew_threshold=0.75
)

# Step 3: Multicollinearity pruning & feature selection
X_train_selected, X_test_selected, selected_cols = fe.select_features_pipeline(
    X_train=X_train_enc,
    y_train_log=y_train_log,
    X_test=X_test_enc,
    corr_threshold=0.90,
    vif_threshold=10.0,
    top_k=60
)

print(f"Selected {len(selected_cols)} features -> X_train_selected: {X_train_selected.shape}")
if X_test_selected is not None:
    print(f"X_test_selected: {X_test_selected.shape}")

c:\Users\Jonathan\.conda\envs\real-estate-env\lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Jonathan\.conda\envs\real-estate-env\lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Jonathan\.conda\envs\real-estate-env\lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Jonathan\.conda\envs\real-estate-env\lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
 

Selected 60 features -> X_train_selected: (1458, 60)
X_test_selected: (1459, 60)


In [4]:
# Step 4: Write CSV files directly to data/processed
saved = fe.save_processed_datasets(
    X_train_selected=X_train_selected,
    y_train_log=y_train_log,
    X_test_selected=X_test_selected,
    output_dir=DATA_PROCESSED_DIR
)

for name, path in saved.items():
    print(f"Created CSV [{name}]: {path} ({os.path.getsize(path):,} bytes)")

Created CSV [X_train_selected]: ..\data\processed\X_train_selected.csv (780,568 bytes)
Created CSV [y_train_log]: ..\data\processed\y_train_log.csv (28,934 bytes)
Created CSV [X_test_selected]: ..\data\processed\X_test_selected.csv (782,588 bytes)
